In [1]:
%pip install -q deltalake requests pandas pyarrow

from pathlib import Path
from typing import Any, Optional
import json

import pandas as pd
import pyarrow as pa
import requests
from IPython.display import display
from deltalake import DeltaTable, write_deltalake
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_LAKE_ROOT = Path('data_lake') if Path('data_lake').exists() else Path('../data_lake')
DEFAULT_COIN_ID = 'bitcoin'
DEFAULT_VS_CURRENCY = 'usd'

Note: you may need to restart the kernel to use updated packages.


# Fase 1: Extraccion de datos con CoinGecko

URL base de la API: `https://api.coingecko.com/api/v3`

## Endpoints seleccionados
- Endpoint incremental: `/coins/{id}/market_chart`
- Endpoint estático: `/coins/{id}`

## Motivo de la eleccion de la API
- Expone datos temporales y metadatos descriptivos dentro de la misma fuente.
- No requiere API key.
- Las respuestas se entregan en JSON y agilizan la conversion a DataFrames de Pandas.

In [2]:
BASE_URL = 'https://api.coingecko.com/api/v3'

def build_session() -> requests.Session:
    session = requests.Session()
    retry_strategy = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=('GET',),
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    session.headers.update({'Accept': 'application/json'})
    return session

def fetch_json(session: requests.Session, endpoint: str, params: Optional[dict[str, Any]] = None, timeout: int = 20) -> dict[str, Any]:
    response = session.get(f'{BASE_URL}{endpoint}', params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    if not payload:
        raise ValueError('The API returned an empty response.')
    return payload

def extract_incremental_data(
    session: requests.Session,
    coin_id: str,
    vs_currency: str = DEFAULT_VS_CURRENCY,
    days: int = 30,
    interval: str = 'daily',
) -> pd.DataFrame:
    raw_data = fetch_json(
        session,
        f'/coins/{coin_id}/market_chart',
        params={
            'vs_currency': vs_currency,
            'days': days,
            'interval': interval,
        },
    )

    prices_df = pd.DataFrame(raw_data.get('prices', []), columns=['timestamp_ms', 'price'])
    market_caps_df = pd.DataFrame(raw_data.get('market_caps', []), columns=['timestamp_ms', 'market_cap'])
    volumes_df = pd.DataFrame(raw_data.get('total_volumes', []), columns=['timestamp_ms', 'total_volume'])

    if prices_df.empty:
        raise ValueError('No temporal data was returned for the requested asset.')

    df = prices_df.merge(market_caps_df, on='timestamp_ms', how='left').merge(volumes_df, on='timestamp_ms', how='left')
    df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit='ms', utc=True)
    df = df.drop(columns=['timestamp_ms']).sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='last')

    for column in ['price', 'market_cap', 'total_volume']:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')

    df['coin_id'] = coin_id
    df['vs_currency'] = vs_currency
    df['extraction_timestamp'] = pd.Timestamp.now(tz='UTC')
    df['date'] = df['timestamp'].dt.strftime('%Y-%m-%d')
    df['year'] = df['timestamp'].dt.year.astype('int64')
    df['month'] = df['timestamp'].dt.month.astype('int64')
    df['day'] = df['timestamp'].dt.day.astype('int64')

    has_intraday_grain = df['timestamp'].dt.floor('h').nunique() > df['date'].nunique()
    if has_intraday_grain:
        df['hour'] = df['timestamp'].dt.hour.astype('int64')

    return df.reset_index(drop=True)

def extract_static_data(
    session: requests.Session,
    coin_id: str,
    vs_currency: str = DEFAULT_VS_CURRENCY,
) -> pd.DataFrame:
    raw_data = fetch_json(
        session,
        f'/coins/{coin_id}',
        params={
            'localization': 'false',
            'tickers': 'false',
            'market_data': 'true',
            'community_data': 'false',
            'developer_data': 'false',
            'sparkline': 'false',
        },
    )

    df = pd.json_normalize(raw_data, sep='_')
    if df.empty:
        raise ValueError('No static metadata was returned for the requested asset.')

    currency_suffix = vs_currency.lower()
    selected_columns = [
        'id',
        'symbol',
        'name',
        'block_time_in_minutes',
        'hashing_algorithm',
        'country_origin',
        'genesis_date',
        'market_cap_rank',
        'coingecko_rank',
        f'market_data_current_price_{currency_suffix}',
        f'market_data_market_cap_{currency_suffix}',
        f'market_data_total_volume_{currency_suffix}',
        f'market_data_fully_diluted_valuation_{currency_suffix}',
        f'market_data_high_24h_{currency_suffix}',
        f'market_data_low_24h_{currency_suffix}',
        'market_data_market_cap_change_percentage_24h',
        'market_data_price_change_percentage_24h',
        'market_data_price_change_percentage_7d',
        'market_data_price_change_percentage_30d',
        f'market_data_ath_{currency_suffix}',
        f'market_data_ath_change_percentage_{currency_suffix}',
        f'market_data_ath_date_{currency_suffix}',
        f'market_data_atl_{currency_suffix}',
        f'market_data_atl_change_percentage_{currency_suffix}',
        f'market_data_atl_date_{currency_suffix}',
        'market_data_circulating_supply',
        'market_data_total_supply',
        'market_data_max_supply',
    ]
    available_columns = [column for column in selected_columns if column in df.columns]
    df = df[available_columns].copy()

    numeric_columns = [
        column
        for column in df.columns
        if column.startswith('market_data_') or column in {'block_time_in_minutes', 'market_cap_rank', 'coingecko_rank'}
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

    for column in [
        'genesis_date',
        f'market_data_ath_date_{currency_suffix}',
        f'market_data_atl_date_{currency_suffix}',
    ]:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors='coerce')

    rename_map = {
        f'market_data_current_price_{currency_suffix}': f'current_price_{currency_suffix}',
        f'market_data_market_cap_{currency_suffix}': f'market_cap_{currency_suffix}',
        f'market_data_total_volume_{currency_suffix}': f'total_volume_{currency_suffix}',
        f'market_data_fully_diluted_valuation_{currency_suffix}': f'fdv_{currency_suffix}',
        f'market_data_high_24h_{currency_suffix}': f'high_24h_{currency_suffix}',
        f'market_data_low_24h_{currency_suffix}': f'low_24h_{currency_suffix}',
        f'market_data_ath_{currency_suffix}': f'ath_{currency_suffix}',
        f'market_data_ath_change_percentage_{currency_suffix}': f'ath_change_pct_{currency_suffix}',
        f'market_data_ath_date_{currency_suffix}': f'ath_date_{currency_suffix}',
        f'market_data_atl_{currency_suffix}': f'atl_{currency_suffix}',
        f'market_data_atl_change_percentage_{currency_suffix}': f'atl_change_pct_{currency_suffix}',
        f'market_data_atl_date_{currency_suffix}': f'atl_date_{currency_suffix}',
        'market_data_market_cap_change_percentage_24h': 'market_cap_change_pct_24h',
        'market_data_price_change_percentage_24h': 'price_change_pct_24h',
        'market_data_price_change_percentage_7d': 'price_change_pct_7d',
        'market_data_price_change_percentage_30d': 'price_change_pct_30d',
        'market_data_circulating_supply': 'circulating_supply',
        'market_data_total_supply': 'total_supply',
        'market_data_max_supply': 'max_supply',
    }
    df = df.rename(columns=rename_map)
    df['coin_id'] = coin_id
    df['vs_currency'] = vs_currency
    df['snapshot_date'] = pd.Timestamp.now(tz='UTC').date().isoformat()
    df['extraction_timestamp'] = pd.Timestamp.now(tz='UTC')
    return df

# Fase 2: Almacenamiento en Delta Lake

- La carga full se escribe con `overwrite` porque representa el estado completo actual del activo.
- La carga incremental se escribe con `MERGE` para evitar duplicados en reejecuciones.
- La tabla incremental se particiona por `coin_id`, `year`, `month`, `day` y `hour` cuando existe granularidad intradiaria.

In [3]:
def table_path(table_name: str) -> Path:
    DATA_LAKE_ROOT.mkdir(parents=True, exist_ok=True)
    return DATA_LAKE_ROOT / table_name

def table_exists(path: Path) -> bool:
    return (path / '_delta_log').exists()

def prepare_for_storage(df: pd.DataFrame) -> pd.DataFrame:
    prepared_df = df.dropna(axis=1, how='all').copy()
    for column in prepared_df.columns:
        if prepared_df[column].map(lambda value: isinstance(value, (list, dict, tuple, set))).any():
            prepared_df[column] = prepared_df[column].apply(
                lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict, tuple, set)) else value
            )
    return prepared_df

def write_full_snapshot(
    df: pd.DataFrame,
    table_name: str,
    partition_by: Optional[list[str]] = None,
) -> Path:
    if df.empty:
        raise ValueError(f'La tabla {table_name} no se puede persistir porque el DataFrame esta vacio.')

    prepared_df = prepare_for_storage(df)
    path = table_path(table_name)
    path.mkdir(parents=True, exist_ok=True)
    write_deltalake(
        str(path),
        pa.Table.from_pandas(prepared_df, preserve_index=False),
        mode='overwrite',
        schema_mode='overwrite',
        partition_by=partition_by,
    )
    return path

def upsert_incremental(
    df: pd.DataFrame,
    table_name: str,
    merge_keys: list[str],
    partition_by: Optional[list[str]] = None,
) -> Path:
    if df.empty:
        raise ValueError(f'La tabla {table_name} no se puede persistir porque el DataFrame esta vacio.')

    prepared_df = prepare_for_storage(df)
    deduped_df = prepared_df.sort_values('extraction_timestamp').drop_duplicates(subset=merge_keys, keep='last').reset_index(drop=True)
    path = table_path(table_name)
    path.mkdir(parents=True, exist_ok=True)
    source_table = pa.Table.from_pandas(deduped_df, preserve_index=False)

    if not table_exists(path):
        write_deltalake(
            str(path),
            source_table,
            mode='overwrite',
            partition_by=partition_by,
        )
        return path

    delta_table = DeltaTable(str(path))
    predicate = ' AND '.join([f'target.{column} = source.{column}' for column in merge_keys])
    (
        delta_table.merge(source_table, predicate=predicate, source_alias='source', target_alias='target')
        .when_matched_update_all()
        .when_not_matched_insert_all()
        .execute()
    )
    return path

In [4]:
def run_pipeline(coin_id: str = DEFAULT_COIN_ID) -> dict[str, Any]:
    session = build_session()

    static_df = extract_static_data(session, coin_id)
    incremental_df = extract_incremental_data(
        session=session,
        coin_id=coin_id,
        vs_currency=DEFAULT_VS_CURRENCY,
        days=30,
        interval='daily',
    )

    static_path = write_full_snapshot(
        static_df,
        table_name='coin_metadata',
        partition_by=['coin_id'],
    )

    incremental_partitions = ['coin_id', 'year', 'month', 'day']
    if 'hour' in incremental_df.columns:
        incremental_partitions.append('hour')

    incremental_path = upsert_incremental(
        incremental_df,
        table_name='coin_market_history',
        merge_keys=['coin_id', 'timestamp'],
        partition_by=incremental_partitions,
    )

    print(f'Full snapshot rows: {len(static_df)} -> {static_path}')
    print(f'Incremental rows: {len(incremental_df)} -> {incremental_path}')

    display(static_df.head())
    display(incremental_df.head())

    return {
        'static_df': static_df,
        'incremental_df': incremental_df,
        'static_path': static_path,
        'incremental_path': incremental_path,
    }

In [5]:
pipeline_result = run_pipeline(DEFAULT_COIN_ID)

static_df = pipeline_result['static_df']
incremental_df = pipeline_result['incremental_df']
static_delta_path = pipeline_result['static_path']
incremental_delta_path = pipeline_result['incremental_path']

print('Delta tables created successfully.')
print(f'Static table path: {static_delta_path}')
print(f'Incremental table path: {incremental_delta_path}')

Full snapshot rows: 1 -> data_lake\coin_metadata
Incremental rows: 31 -> data_lake\coin_market_history


,id,symbol,name,block_time_in_minutes,hashing_algorithm,country_origin,genesis_date,market_cap_rank,current_price_usd,market_cap_usd,...,atl_usd,atl_change_pct_usd,atl_date_usd,circulating_supply,total_supply,max_supply,coin_id,vs_currency,snapshot_date,extraction_timestamp
0,bitcoin,btc,Bitcoin,10,SHA-256,,2009-01-03,1,71422,1429145040207,...,67.81,105227.75107,NaT,20002450.0,20002450.0,21000000.0,bitcoin,usd,2026-03-15,2026-03-15 16:24:59.677432+00:00


,price,market_cap,total_volume,timestamp,coin_id,vs_currency,extraction_timestamp,date,year,month,day,hour
0,68838.874905,1.376029e+12,4.330735e+10,2026-02-14 00:00:00+00:00,bitcoin,usd,2026-03-15 16:24:59.856190+00:00,2026-02-14,2026,2,14,0
1,69765.596388,1.395186e+12,3.888673e+10,2026-02-15 00:00:00+00:00,bitcoin,usd,2026-03-15 16:24:59.856190+00:00,2026-02-15,2026,2,15,0
2,68716.583375,1.373547e+12,4.413954e+10,2026-02-16 00:00:00+00:00,bitcoin,usd,2026-03-15 16:24:59.856190+00:00,2026-02-16,2026,2,16,0
3,68907.783536,1.377440e+12,3.678921e+10,2026-02-17 00:00:00+00:00,bitcoin,usd,2026-03-15 16:24:59.856190+00:00,2026-02-17,2026,2,17,0
4,67489.455403,1.349285e+12,3.833900e+10,2026-02-18 00:00:00+00:00,bitcoin,usd,2026-03-15 16:24:59.856190+00:00,2026-02-18,2026,2,18,0


Delta tables created successfully.
Static table path: data_lake\coin_metadata
Incremental table path: data_lake\coin_market_history


## Resumen para entrega

- El notebook usa una extracción incremental para el histórico de mercado y una extracción full para el snapshot del activo.
- Los datos se convierten a DataFrames de Pandas y se guardan en Delta Lake en directorios separados.
- La carga full utiliza `overwrite` y la carga incremental utiliza `MERGE` idempotente.
- La tabla incremental queda particionada por `coin_id`, `year`, `month`, `day` y `hour` cuando corresponde.